# 06. Simple(r) Analysis + Charts for Presentation/Article

Notebook 05 contains much more in-depth charts and analysis, meant to be read by a data-familiar audience. This notebook contains only the charts that appear in the article and slide deck (plus setup, data loading, and checks).

| # | Chart | Used in | Reads |
|---|---|---|---|
| 1 | The parties swap places when the White House changes hands (monthly sentiment by party + inflation panel) | Article lede · slide 7 | `fct_say_vs_do_monthly` |
| 2 | The share who say the economy improved follows the president's party (CES, annual) | Article | `fct_party_ratings` |
| 3 | The flip starts at the election (UMich, Jun 2024–Sep 2025) | Article · slide 9 | `fct_say_vs_do_monthly`, `fct_swing_timing_bootstrap` |
| 4 | Presidential elections move the partisan gap; midterms don't | Article · slide 10 (left) | `fct_midterm_placebo` |
| 5 | The partisan gap is at least twice as big on "the economy" as on people's own finances (ANES) | Article · slide 10 (right) | `fct_pocketbook_gap_bootstrap`, `fct_pocketbook_gap_model` |
| 6 | Sentiment swung; spending growth hardly moved | Article · slide 11 | `fct_say_vs_do_monthly`, `fct_say_vs_do_correlations` |

## 0. Setup

In [0]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
import textwrap
from pathlib import Path

# ── Analysis and plotting (pre-installed on Databricks serverless) ───────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

# ── Where the tables live ────────────────────────────────────────────────────
CATALOG = "workspace"
SCHEMA = "public_opinion_economy_case_study"

# `spark` exists in every Databricks notebook. If it doesn't, we're running locally and read parquet instead.
try:
    spark  # noqa: F821
    IN_DATABRICKS = True
except NameError:
    IN_DATABRICKS = False

LOCAL_MARTS_DIR = Path(os.environ.get("LOCAL_MARTS_DIR", "marts"))   # one parquet file or folder per table

# ── Where the charts go ──────────────────────────────────────────────────────
# Databricks: the same `outputs` volume notebook 05 uses, in its own subfolder so the two sets never mix.
# Local: ./outputs/public_charts next to the notebook.
if IN_DATABRICKS:
    spark.sql(f"USE CATALOG {CATALOG}")                                   # noqa: F821
    spark.sql(f"USE SCHEMA {SCHEMA}")                                     # noqa: F821
    spark.sql("CREATE VOLUME IF NOT EXISTS outputs COMMENT 'Charts and tables exported by notebooks 05 and 06'")  # noqa: F821
    OUTPUT_DIR = Path(f"/Volumes/{CATALOG}/{SCHEMA}/outputs/public_charts")
else:
    OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "outputs")) / "public_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running in {'Databricks' if IN_DATABRICKS else 'local mode'}; charts -> {OUTPUT_DIR}")

## 1. Style

In [0]:
# ── Palette (same hex values as the article's light theme) ───────────────────
REP, DEM, IND = "#B8322A", "#2F6FB3", "#A87710"     # party colors: semantic and fixed, never reassigned
NEUTRAL = "#6B7280"                                  # non-partisan lines (inflation, spending)
STRONG, SOFT = "#343A44", "#A9AFB8"                  # two-category bars (presidential/midterm, national/personal)
INK, MUTED, GRID = "#1D2026", "#5B616B", "#E4E5E1"
PAGE = "#FBFBF9"                                     # the article's background, so PNGs sit on the page seamlessly
TINT = {"Republican": "#F5E8E5", "Democrat": "#E5EDF6"}   # presidency shading (very light, behind everything)

PARTY = {  # key → (label, color); one place to rename or recolor a series
    "rep": ("Republicans", REP),
    "dem": ("Democrats", DEM),
    "ind": ("Independents", IND),
}

# ── Font: the article's face if available, else a safe default ───────────────
FONT = "Libre Franklin" if any("Libre Franklin" in f.name for f in font_manager.fontManager.ttflist) else "DejaVu Sans"

# ── Two output sizes. Slide text is larger so it reads from the back of a room. ─
SIZES = {
    "article": {"figsize": (9.0, 5.6), "base": 10.5},
    "slide": {"figsize": (13.33, 7.5), "base": 15},
}


def style(base):
    """rcParams for one output size. Everything scales from `base` so the hierarchy stays the same at both sizes."""
    return {
        "font.family": FONT,
        "font.size": base,
        "figure.facecolor": PAGE,
        "axes.facecolor": PAGE,
        "savefig.facecolor": PAGE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": False,           # gridlines carry the scale; no heavy axis line
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "grid.linewidth": 1,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": base * 0.9,
        "ytick.labelsize": base * 0.9,
        "ytick.left": False,
        "lines.linewidth": 2.2,
        "lines.solid_capstyle": "round",
        "savefig.dpi": 200,
    }


def frame(fig, title, subtitle, src, base, left, right, above_axes=0.0, below_axes=1.0):
    """Newsroom frame: takeaway title (the finding), plain subtitle (what is measured) and a source line, then fit
    the plot area between them. Text is wrapped to the figure's real width and the margins are computed from the
    number of wrapped lines, so a longer title or the larger slide fonts never overlap the chart or run off the edge.
    `above_axes` / `below_axes`: extra room in multiples of `base` for labels drawn just outside the axes
    (president names above the lede, tick labels below)."""
    W, H = fig.get_figwidth() * 72, fig.get_figheight() * 72          # figure size in points
    t_fs, s_fs, c_fs = base * 1.45, base * 0.95, base * 0.75
    usable = W * 0.97
    t_lines = textwrap.wrap(title, int(usable / (t_fs * 0.6)))          # ~0.6 em per bold character
    s_lines = textwrap.wrap(subtitle, int(usable / (s_fs * 0.53)))
    c_lines = textwrap.wrap(src, int(usable / (c_fs * 0.53)))
    x0, y = 0.012, H - 8
    fig.text(x0, y / H, "\n".join(t_lines), ha="left", va="top", fontsize=t_fs, fontweight="bold", color=INK,
             linespacing=1.15)
    y -= len(t_lines) * t_fs * 1.15 * 1.17 + base * 0.6
    fig.text(x0, y / H, "\n".join(s_lines), ha="left", va="top", fontsize=s_fs, color=MUTED, linespacing=1.3)
    y -= len(s_lines) * s_fs * 1.3 * 1.17 + base * (1.4 + above_axes)
    fig.text(x0, 6 / H, "\n".join(c_lines), ha="left", va="bottom", fontsize=c_fs, color=MUTED, linespacing=1.3)
    bottom = 6 + len(c_lines) * c_fs * 1.3 * 1.17 + base * (1.0 + below_axes * 1.4)
    fig.subplots_adjust(left=left, right=right, top=y / H, bottom=bottom / H)


def end_labels(ax, x, values, base, min_gap):
    """Direct labels at the right end of each line: a short colored tick for identity, the name in ink.
    `values` = {key: last y}. Labels closer than `min_gap` (data units) are nudged apart so they never overlap."""
    items = sorted(values.items(), key=lambda kv: kv[1], reverse=True)
    placed = []
    for key, y in items:
        y_lab = y if not placed or placed[-1] - y >= min_gap else placed[-1] - min_gap
        placed.append(y_lab)
        label, color = PARTY.get(key, (key, NEUTRAL))
        ax.annotate("━ ", (x, y_lab), xytext=(8, 0), textcoords="offset points", va="center", color=color,
                    fontsize=base * 0.9, annotation_clip=False)
        ax.annotate(label, (x, y_lab), xytext=(24, 0), textcoords="offset points", va="center", color=INK,
                    fontsize=base * 0.95, fontweight="bold", annotation_clip=False)
        ax.plot([x], [y], "o", color=color, markersize=base * 0.45, markeredgecolor=PAGE, markeredgewidth=1.5,
                zorder=5)


def event_line(ax, when, label, base, y_text, ha="left"):
    """Dashed vertical marker for an election or inauguration, labeled at the bottom of the panel."""
    ax.axvline(when, color="#8A9099", lw=1.2, ls=(0, (4, 4)), zorder=1)
    offset = 6 if ha == "left" else -6
    ax.annotate(label, (when, y_text), xytext=(offset, 0), textcoords="offset points", ha=ha, va="bottom",
                fontsize=base * 0.9, fontweight="bold", color=INK)


def save(fig, name, size):
    """PNG for the web page and slides, SVG for editing; same stem, size in the file name."""
    for ext in ("png", "svg"):
        fig.savefig(OUTPUT_DIR / f"{name}_{size}.{ext}")
    plt.show() if size == "article" else plt.close(fig)   # show the article version inline; keep output short

## 2. Load the marts

In [0]:
EXPECTED_ROWS = {
    "fct_say_vs_do_monthly": 128,          # monthly party sentiment, inflation, real spending (charts 1, 3, 6)
    "fct_party_ratings": 188,              # CES % better by party and year (chart 2)
    "fct_swing_timing_bootstrap": 21,      # share of the swing done by election / inauguration (chart 3 subtitle)
    "fct_midterm_placebo": 21,             # gap change after each election, with noise band (chart 4)
    "fct_pocketbook_gap_bootstrap": 12,    # ANES national vs personal partisan gap (chart 5)
    "fct_pocketbook_gap_model": 18,        # same gaps with income/job/homeownership controls (chart 5 note)
    "fct_say_vs_do_correlations": 16,      # sentiment vs spending correlations (chart 6 subtitle)
}


def load(name: str) -> pd.DataFrame:
    """Read one mart from Unity Catalog (Databricks) or a local parquet copy."""
    if IN_DATABRICKS:
        pdf = spark.table(f"{CATALOG}.{SCHEMA}.{name}").toPandas()                     # noqa: F821
    else:
        path = LOCAL_MARTS_DIR / name
        path = path if path.exists() else LOCAL_MARTS_DIR / f"{name}.parquet"
        pdf = pd.read_parquet(path)
    # Spark DATE columns arrive as python `datetime.date`; make them Timestamps once so time axes just work.
    for col in ("month", "first_month", "last_month"):
        if col in pdf.columns and not pd.api.types.is_numeric_dtype(pdf[col]):
            pdf[col] = pd.to_datetime(pdf[col])
    return pdf


M = {name: load(name) for name in EXPECTED_ROWS}
load_report = pd.DataFrame([{"table": n, "rows": len(df), "expected": EXPECTED_ROWS[n],
                             "matches": len(df) == EXPECTED_ROWS[n]} for n, df in M.items()])
display(load_report)
if not load_report["matches"].all():
    print("WARNING: row counts differ from the handoff (expected if new months were loaded). Check section 3.")


def one(df: pd.DataFrame, col: str, **filters):
    """Exactly one value after equality filters; fail loudly if the filter isn't unique.
    A silent .iloc[0] on a non-unique filter is how wrong numbers end up in print."""
    sub = df
    for k, v in filters.items():
        sub = sub[sub[k] == v]
    if len(sub) != 1:
        raise ValueError(f"{col} with {filters}: expected 1 row, found {len(sub)}")
    return sub[col].iloc[0]

## 3. Chart data and QA checks

In [0]:
# ── Monthly series (charts 1, 3, 6) ──────────────────────────────────────────
monthly = (M["fct_say_vs_do_monthly"]
           .rename(columns={"sentiment_rep": "rep", "sentiment_dem": "dem", "sentiment_ind": "ind",
                            "inflation_yoy_pct": "inflation", "pce_real_yoy_pct": "spending"})
           [["month", "white_house_party", "rep", "dem", "ind", "inflation", "spending"]]
           .sort_values("month").set_index("month"))
# The UMich party series starts in April 2017; earlier rows have inflation but no party sentiment.
party_start = monthly[["rep", "dem", "ind"]].dropna().index.min()

# Presidencies, derived from the mart's White House column (a new term = a change in that column). Only the
# president's surname is hand-entered, because the mart stores the party, not the name.
PRESIDENT_BY_TERM_START = {2017: "Trump", 2021: "Biden", 2025: "Trump"}
wh = monthly["white_house_party"]
starts = wh.index[(wh != wh.shift()).to_numpy()]
terms = [(s, (starts[i + 1] if i + 1 < len(starts) else monthly.index.max()), wh[s]) for i, s in enumerate(starts)]

# Event dates used on the timing and spending charts (the 2024 transition)
ELECTION_2024, INAUGURATION_2025 = pd.Timestamp("2024-11-01"), pd.Timestamp("2025-01-01")

# ── CES annual (chart 2): % saying the economy got better, leaners counted with their party ──
ces = (M["fct_party_ratings"]
       .query("source == 'CES' and party_definition == 'Leaners with parties'")
       .pivot(index="survey_year", columns="party_group", values="pct_better")
       .rename(columns={"Republican": "rep", "Democrat": "dem", "Independent": "ind"})[["rep", "dem", "ind"]])

# ── Election shifts (chart 4): UMich, three months after each election ───────
placebo = M["fct_midterm_placebo"]
shifts = (placebo[placebo["source"].str.startswith("UMich") & (placebo["checkpoint_months_from_election"] == 3)]
          .assign(year=lambda d: d["event_label"].str[:4].astype(int),
                  kind=lambda d: np.where(d["event_type"].str.startswith("Midterm"), "Midterm", "Presidential"))
          .sort_values("year")
          [["year", "kind", "event_label", "winning_party", "chambers_flipped",
            "winner_minus_loser_gap_change", "gap_change_noise_band"]])
NOISE_BAND = shifts["gap_change_noise_band"].iloc[0]       # same band for every event (one noise estimate per series)

# ── ANES gaps (chart 5): main spec = "don't know" excluded, leaners with parties ─
anes = (M["fct_pocketbook_gap_bootstrap"].query("is_main_spec")
        .set_index("survey_year")[["national_gap_pts", "personal_gap_pts", "national_to_personal_ratio"]])
gap_model = M["fct_pocketbook_gap_model"].query(
    "outcome_spec == 'Main: do not know excluded' and party_definition == 'Leaners with parties'")
controls_diff = (gap_model[gap_model["term"].str.startswith("national_minus_personal")]
                 .pivot(index="survey_year", columns="with_controls", values="estimate_pp")
                 .rename(columns={False: "no_controls", True: "with_controls"}))

# ── Swing timing and say-vs-do (subtitles of charts 3 and 6) ─────────────────
swing = M["fct_swing_timing_bootstrap"]
corr = M["fct_say_vs_do_correlations"]
party_corr_max = (corr[(corr["sample"] == "All months") & corr["say_measure"].isin(
    ["Republican sentiment", "Democratic sentiment", "Independent sentiment"]) &
    (corr["do_measure"] == "Real PCE growth (% m/m)")]["r"].abs().max())

# ── Every number shown on or beside a chart, against its published value ────
in_window = monthly.loc["2024-06":"2025-09", "spending"]
gap_by_year = (monthly["rep"] - monthly["dem"]).groupby(monthly.index.year).mean()
published = [
    # (where it appears, computed value, published value, decimals)
    ("Ch1/lede: Rep−Dem gap, 2026 average", gap_by_year[2026], 53, 0),
    ("Ch1/lede: inflation peak (%)", monthly["inflation"].max(), 9.1, 1),
    ("Ch2: CES Republicans 2024 (%)", ces.loc[2024, "rep"], 5, 0),
    ("Ch2: CES Republicans 2025 (%)", ces.loc[2025, "rep"], 52, 0),
    ("Ch2: CES Democrats 2024 (%)", ces.loc[2024, "dem"], 52, 0),
    ("Ch2: CES Democrats 2025 (%)", ces.loc[2025, "dem"], 5, 0),
    ("Ch3: Rep share of swing done by inauguration, 2024–25 (%)",
     one(swing, "pct_swing_done_at_takeover", event_key="pres-2024", series="Republican"), 70, 0),
    ("Ch3: Rep share done by election month, 2024–25 (%)",
     one(swing, "pct_swing_done_at_election", event_key="pres-2024", series="Republican"), 16, 0),
    ("Ch3: Dem share done by inauguration, 2024–25 (%)",
     one(swing, "pct_swing_done_at_takeover", event_key="pres-2024", series="Democrat"), 35, 0),
    ("Ch3: Rep share done by inauguration, 2020–21 (%)",
     one(swing, "pct_swing_done_at_takeover", event_key="pres-2020", series="Republican"), 59, 0),
    ("Ch3: Dem share done by inauguration, 2020–21 (%)",
     one(swing, "pct_swing_done_at_takeover", event_key="pres-2020", series="Democrat"), 56, 0),
    *[(f"Ch4: gap shift, {r.event_label} (pts)", r.winner_minus_loser_gap_change, pub, 1)
      for r, pub in zip(shifts.itertuples(), [3.1, 57.2, 2.2, 61.9])],
    ("Ch4: noise band (pts)", NOISE_BAND, 6.5, 1),
    *[(f"Ch5: ANES national gap {y} (pts)", anes.loc[y, "national_gap_pts"], pub, 0)
      for y, pub in [(2016, 29), (2020, 39), (2024, 50)]],
    *[(f"Ch5: ANES personal gap {y} (pts)", anes.loc[y, "personal_gap_pts"], pub, 0)
      for y, pub in [(2016, 13), (2020, 11), (2024, 25)]],
    *[(f"Ch5: national − personal with controls {y} (pts)", controls_diff.loc[y, "with_controls"], pub, 0)
      for y, pub in [(2016, 16), (2020, 28), (2024, 25)]],
    ("Ch6: max |r|, party sentiment vs spending", party_corr_max, 0.10, 2),
    ("Ch6: spending growth low, Jun 2024–Sep 2025 (%)", in_window.min(), 2.4, 1),
    ("Ch6: spending growth high, Jun 2024–Sep 2025 (%)", in_window.max(), 3.7, 1),
]
checks = pd.DataFrame(published, columns=["where", "computed", "published", "decimals"])
checks["matches"] = [round(c, d) == round(p, d) for c, p, d in
                     zip(checks["computed"], checks["published"], checks["decimals"])]
display(checks.drop(columns="decimals").style.format({"computed": "{:.2f}", "published": "{:g}"}))
print(f"{checks['matches'].sum()} of {len(checks)} published numbers match the marts.")
if not checks["matches"].all():
    print("MISMATCH: update the article/deck text (or find the upstream change) before re-exporting.")

## 4. Chart 1: the lede
**Finding:** each time the White House changes hands, the Republican and Democratic lines cross. Inflation, in its own panel, doesn't line up with either crossing.

In [0]:
def chart_lede(size):
    cfg = SIZES[size]
    base = cfg["base"]
    with plt.rc_context(style(base)):
        fig, (ax, ax2) = plt.subplots(2, 1, figsize=cfg["figsize"], sharex=True,
                                      gridspec_kw={"height_ratios": [3.2, 1], "hspace": 0.18})
        d = monthly.loc[party_start:]

        # Presidency shading (both panels) with the president's name above the top panel
        for start, end, party in terms:
            if end < party_start:
                continue
            for a in (ax, ax2):
                a.axvspan(max(start, party_start), end, color=TINT[party], lw=0, zorder=0)
            mid = max(start, party_start) + (end - max(start, party_start)) / 2
            ax.text(mid, 1.02, PRESIDENT_BY_TERM_START.get(start.year, ""), transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=base * 0.95, fontweight="bold", color=INK)

        # Independents first so the two party lines draw on top of them
        for key in ("ind", "dem", "rep"):
            ax.plot(d.index, d[key], color=PARTY[key][1], zorder=3)
        ax.set_ylim(20, 140)
        ax.set_yticks([20, 60, 100, 140])
        end_labels(ax, d.index[-1], {k: d[k].iloc[-1] for k in ("rep", "dem", "ind")}, base, min_gap=9)

        ax2.plot(d.index, d["inflation"], color=NEUTRAL, zorder=3)
        ax2.set_ylim(0, 10)
        ax2.set_yticks([0, 5, 10])
        ax2.set_yticklabels(["0", "5", "10%"])
        ax2.text(0.005, 0.97, "Inflation, year over year", transform=ax2.transAxes, va="top",
                 fontsize=base * 0.9, fontweight="bold", color=INK)
        ax2.xaxis.set_major_locator(mdates.YearLocator(base=1 if size == "slide" else 2))
        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax2.tick_params(axis="x", length=0)
        ax.set_xlim(party_start, d.index[-1])

        frame(fig, "Party sentiment swaps places when the White House changes hands",
              "Consumer sentiment by party. Inflation is shown below on its own scale.",
              f"Source: University of Michigan Surveys of Consumers (sentiment by party, {party_start:%b %Y} to "
              f"{d.index[-1]:%b %Y}); BLS consumer price index, year-over-year change.",
              base, left=0.06, right=0.84, above_axes=1.6)
        save(fig, "01_lede_parties_swap_places", size)


for size in SIZES:
    chart_lede(size)

## 5. Chart 2: the CES annual series
**Finding:** the share of each party saying the economy improved jumps or collapses in the first survey after a new
president takes office. Independents stay in between.

The CES is fielded in the fall, so "takes office" markers sit halfway between survey years. Values are labeled only at
the end of each line, the number the article quotes.

In [0]:
def chart_ces(size):
    cfg = SIZES[size]
    base = cfg["base"]
    with plt.rc_context(style(base)):
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        for when, who in [(2016.5, "Trump"), (2020.5, "Biden"), (2024.5, "Trump")]:
            ax.axvline(when, color="#8A9099", lw=1.2, ls=(0, (4, 4)), zorder=1)
            ax.text(when + 0.08, 101, f"{who} takes office", ha="left", va="bottom", fontsize=base * 0.85,
                    fontweight="bold", color=INK)
        for key in ("ind", "dem", "rep"):
            ax.plot(ces.index, ces[key], color=PARTY[key][1], marker="o", markersize=base * 0.4,
                    markeredgecolor=PAGE, markeredgewidth=1.2, zorder=3)
        last = ces.index.max()
        # End labels carry the value the article quotes ("Republicans 52%")
        placed = []
        for key in sorted(("rep", "dem", "ind"), key=lambda k: -ces.loc[last, k]):
            y = ces.loc[last, key]
            y_lab = y if not placed or placed[-1] - y >= 7 else placed[-1] - 7
            placed.append(y_lab)
            ax.annotate(f"{PARTY[key][0]} {y:.0f}%", (last, y_lab), xytext=(10, 0), textcoords="offset points",
                        va="center", fontsize=base * 0.95, fontweight="bold", color=INK, annotation_clip=False)
        ax.set_ylim(0, 100)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_yticklabels(["0", "25", "50", "75", "100%"])
        ax.set_xticks(ces.index)
        ax.tick_params(axis="x", length=0)
        ax.set_xlim(ces.index.min() - 0.3, last + 0.2)

        frame(fig, "The share who say the economy improved follows the president's party",
              "Percent saying the national economy got better over the past year, by party (independents who lean "
              "toward a party are counted with that party)",
              f"Source: Cooperative Election Study cumulative file, {ces.index.min()}–{last}, weighted. "
              "Each year's survey is fielded in the fall.",
              base, left=0.07, right=0.8, above_axes=1.6)
        save(fig, "02_ces_share_saying_better", size)


for size in SIZES:
    chart_ces(size)

## 6. Chart 3: when the flip happens
**Finding:** Republicans' outlook started climbing in the month of the 2024 election, and most of the move was done
by the inauguration. Democrats held until December, then slid into the spring.

The subtitle's percentages are read from `fct_swing_timing_bootstrap` (the bootstrap intervals behind them are in
notebook 05; the article keeps them out of the chart).

In [0]:
def chart_timing(size):
    cfg = SIZES[size]
    base = cfg["base"]
    rep_e = one(swing, "pct_swing_done_at_election", event_key="pres-2024", series="Republican")
    rep_t = one(swing, "pct_swing_done_at_takeover", event_key="pres-2024", series="Republican")
    with plt.rc_context(style(base)):
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        d = monthly.loc["2024-06":"2025-09"]
        for key in ("ind", "dem", "rep"):
            ax.plot(d.index, d[key], color=PARTY[key][1], zorder=3)
        ax.set_ylim(30, 100)
        ax.set_yticks([30, 50, 70, 90])
        event_line(ax, ELECTION_2024, "Election", base, 32, ha="right")
        event_line(ax, INAUGURATION_2025, "Inauguration", base, 32, ha="left")
        end_labels(ax, d.index[-1], {k: d[k].iloc[-1] for k in ("rep", "dem", "ind")}, base, min_gap=5)
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
        ax.tick_params(axis="x", length=0)
        ax.set_xlim(d.index[0], d.index[-1])

        frame(fig, "Outlooks started changing in the month of the 2024 election",
              f"Consumer sentiment by party, University of Michigan index.",
              "Source: University of Michigan Surveys of Consumers",
              base, left=0.06, right=0.82)
        save(fig, "03_flip_starts_at_election", size)


for size in SIZES:
    chart_timing(size)

## 7. Chart 4: presidential elections vs midterms
**Finding:** after the 2020 and 2024 presidential elections, the gap between the winning and losing party's sentiment
moved by about 60 points. After the 2018 and 2022 midterms, when the House flipped, it moved 2 to 3 points.

The shaded band is the series' own month-to-month noise (from the mart, not hand-entered), so a reader can see the
midterm bars sit inside "could be chance". Bars are neutral grey: this chart compares kinds of election, not parties.

In [0]:
def chart_midterms(size):
    cfg = SIZES[size]
    base = cfg["base"]
    with plt.rc_context(style(base)):
        fig, ax = plt.subplots(figsize=(cfg["figsize"][0], cfg["figsize"][1] * 0.8))
        rows = shifts.reset_index(drop=True)
        y = np.arange(len(rows))[::-1]                                      # first election at the top
        ax.axvspan(0, NOISE_BAND, color=GRID, lw=0, zorder=0)
        ax.text(NOISE_BAND / 2, len(rows) - 0.45, "Normal\nwobble", ha="center", va="bottom", fontsize=base * 0.75,
                color=MUTED)
        colors = [STRONG if k == "Presidential" else SOFT for k in rows["kind"]]
        ax.barh(y, rows["winner_minus_loser_gap_change"], color=colors, height=0.6, zorder=2)
        for yi, r in zip(y, rows.itertuples()):
            ax.text(max(r.winner_minus_loser_gap_change, NOISE_BAND) + 0.8, yi, f"{r.winner_minus_loser_gap_change:.1f}",
                    va="center", fontsize=base, fontweight="bold", color=INK)
        what = [f"{r.year} {r.kind.lower()}\n" +
                (f"White House to {r.winning_party}s" if r.kind == "Presidential" else f"House to {r.winning_party}s")
                for r in rows.itertuples()]
        ax.set_yticks(y)
        ax.set_yticklabels(what, fontsize=base * 0.9, color=INK)
        ax.grid(axis="y", visible=False)
        ax.grid(axis="x", visible=True)
        ax.set_xlim(0, 70)
        ax.set_xticks([0, 20, 40, 60])
        ax.tick_params(axis="both", length=0)
        ax.set_ylim(-0.6, len(rows) - 0.1)

        frame(fig, "Presidential elections move the partisan gap. Midterms don't.",
              "Change in the gap between the winning and losing party's sentiment, three months after each "
              "election, in index points. Dark bars: presidential elections; light bars: midterms.",
              f"Source: University of Michigan Surveys of Consumers. Grey band: about {NOISE_BAND:.1f} points, how "
              "far the gap can move by chance, judged from the series' own month-to-month variation.",
              base, left=0.25, right=0.96, above_axes=1.2)
        save(fig, "04_presidential_vs_midterm", size)


for size in SIZES:
    chart_midterms(size)

## 8. Chart 5: the respondent's own wallet vs "the economy"
**Finding:** in every ANES year, the partisan gap on the national economy is at least twice the gap on people's own
finances. Adding income, employment, homeownership and demographic controls doesn't change the difference; that's
stated in the source line, with the numbers read from `fct_pocketbook_gap_model`.

Values are labeled on every bar: there are only six, and the reader needs the numbers to see "twice".

In [0]:
def chart_wallet(size):
    cfg = SIZES[size]
    base = cfg["base"]
    with plt.rc_context(style(base)):
        fig, ax = plt.subplots(figsize=(cfg["figsize"][0] * (0.8 if size == "article" else 1), cfg["figsize"][1]))
        x = np.arange(len(anes))
        w = 0.36
        bars = [("national_gap_pts", "The national economy", STRONG, -w / 2 - 0.01),
                ("personal_gap_pts", "Their own finances", SOFT, w / 2 + 0.01)]   # small gap between paired bars
        for col, label, color, dx in bars:
            ax.bar(x + dx, anes[col], width=w, color=color, label=label, zorder=2)
            for xi, v in zip(x + dx, anes[col]):
                ax.text(xi, v + 1, f"{v:.0f}", ha="center", va="bottom", fontsize=base, fontweight="bold", color=INK)
        ax.set_xticks(x)
        ax.set_xticklabels(anes.index, fontsize=base, fontweight="bold", color=INK)
        ax.tick_params(axis="x", length=0)
        ax.set_ylim(0, 60)
        ax.set_yticks([0, 20, 40, 60])
        ax.set_yticklabels(["0", "20", "40", "60 pts"])
        ax.legend(loc="lower left", ncols=2, fontsize=base * 0.9, handlelength=1, handleheight=1, frameon=False,
                  bbox_to_anchor=(0, 1.0), borderaxespad=0.3)

        diffs = ", ".join(f"{v:.0f}" for v in controls_diff["with_controls"])
        frame(fig, 'The partisan gap is at least twice as big on "the economy" as on people\'s own finances',
              "Percentage-point gap between the out-party and the president's party in the share saying things got "
              "worse over the past year",
              f"Source: American National Election Studies cumulative file, {', '.join(map(str, anes.index))}, "
              f"weighted; excludes \"don't know\". With income, employment, homeownership and demographic "
              f"controls, the national-minus-personal difference is {diffs} points.",
              base, left=0.08, right=0.97, above_axes=1.8)
        save(fig, "05_wallet_vs_economy", size)


for size in SIZES:
    chart_wallet(size)

## 9. Chart 6: what people say vs what they spend
**Finding:** party sentiment crossed over in a few months; real consumer spending growth stayed in its usual range.

Two panels on one time axis rather than two lines on two y-axes. The subtitle's correlation is read from
`fct_say_vs_do_correlations` (monthly changes, all months), so the claim on the chart and in the text can't drift.

In [0]:
def chart_spending(size):
    cfg = SIZES[size]
    base = cfg["base"]
    with plt.rc_context(style(base)):
        fig, (ax, ax2) = plt.subplots(2, 1, figsize=cfg["figsize"], sharex=True,
                                      gridspec_kw={"height_ratios": [2.2, 1], "hspace": 0.28})
        d = monthly.loc["2024-01":]
        for key in ("dem", "rep"):
            ax.plot(d.index, d[key], color=PARTY[key][1], zorder=3)
        ax.set_ylim(20, 100)
        ax.set_yticks([20, 60, 100])
        ax.set_title("Sentiment (UMich index)", loc="left", fontsize=base * 0.95, fontweight="bold", color=INK)
        end_labels(ax, d.index[-1], {k: d[k].iloc[-1] for k in ("rep", "dem")}, base, min_gap=8)

        ax2.plot(d.index, d["spending"], color=NEUTRAL, zorder=3)
        ax2.set_ylim(0, 6)
        ax2.set_yticks([0, 3, 6])
        ax2.set_yticklabels(["0", "3", "6%"])
        ax2.set_title("Real consumer spending, % change from a year earlier", loc="left", fontsize=base * 0.95,
                      fontweight="bold", color=INK)
        for a in (ax, ax2):
            a.axvline(ELECTION_2024, color="#8A9099", lw=1.2, ls=(0, (4, 4)), zorder=1)
            a.axvline(INAUGURATION_2025, color="#8A9099", lw=1.2, ls=(0, (4, 4)), zorder=1)
        ax.annotate("Election", (ELECTION_2024, 22), xytext=(-6, 0), textcoords="offset points", ha="right",
                    va="bottom", fontsize=base * 0.85, fontweight="bold", color=INK)
        ax.annotate("Inauguration", (INAUGURATION_2025, 22), xytext=(6, 0), textcoords="offset points",
                    va="bottom", fontsize=base * 0.85, fontweight="bold", color=INK)
        ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
        ax2.tick_params(axis="x", length=0)
        ax2.set_xlim(d.index[0] - pd.Timedelta(days=25), d.index[-1])

        frame(fig, "Sentiment swung. Spending growth hardly moved.",
              f"Across 2017–{monthly.index.max():%Y}, monthly changes in either party's sentiment and in real "
              f"spending have a correlation of {party_corr_max:.2f} or less.",
              "Source: University of Michigan Surveys of Consumers; Bureau of Economic Analysis real personal "
              "consumption expenditures (PCEC96). Dashed lines: Nov 2024 election, Jan 2025 inauguration.",
              base, left=0.06, right=0.84, above_axes=1.4)
        save(fig, "06_say_vs_spend", size)


for size in SIZES:
    chart_spending(size)